## In-Class Lecture 36-1 - Mission Clock

In this lecture, you will write the four functions at the center of a spacecraft's timekeeping: elapsed time, stale-data rejection, a safe read of a shared counter, and conversion to CCSDS-style time. A test harness checks each one against the cases that break naive code, including the counter wraparound and a timer interrupt at the worst possible moment.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass36
    cd inclass36

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec36/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    Makefile  mission_clock.c  mission_clock.h  test_clock.c

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - Why does a signed 32-bit count of hundredths of a second fail after 248.6 days, and why is the program's behavior after that point not even defined by C?</font>

<font color = "red">Class Introduction Question #2 - Explain why <code>now - start >= TIMEOUT_MS</code> gives the right answer across a wraparound, while <code>now >= start + TIMEOUT_MS</code> does not.</font>

<font color = "red">Class Introduction Question #3 - Method 2 in <code>tick_drift.c</code> uses only integers, but still drifts by more than 5 seconds per day. Why, and how does Method 3 avoid it?</font>

<font color = "red">Class Introduction Question #4 - What is a torn read? Using <code>torn_read.c</code>, explain step by step how <code>now_unsafe()</code> returned a time 49.7 days in the past.</font>

<font color = "red">Class Introduction Question #5 - How does <code>now_safe()</code> detect a torn read? Why does our version use a <code>for</code> loop with <code>MAX_TRIES</code> instead of the <code>do-while</code> loop on Ama's slide?</font>

<font color = "red">Class Introduction Question #6 - Both halves of the counter are <code>volatile</code>. Why was that not enough to prevent the torn read, and what would be?</font>

### The Files

<ul>
    <li><code>mission_clock.h</code> - the <code>flight_time</code> structure, the constants, and the four functions you will write. Read it first.</li>
    <li><code>mission_clock.c</code> - your code. Each function compiles but returns a placeholder.</li>
    <li><code>test_clock.c</code> - 15 tests and the <b>simulated hardware</b>: the two halves of the 64-bit millisecond counter, and a timer interrupt that the tests can trigger. You do not need to change this file.</li>
</ul>

Run the tests before you write anything:

    make test

Three tests pass by coincidence, because the placeholders happen to return the right answer for them. Twelve fail. Your job is to make all 15 pass. Run <code>make test</code> after every step.

### Step 1 - <code>elapsed_ms</code> (Tests 1 to 3)

Return the number of milliseconds from <code>start</code> to <code>now</code>. Test 2 starts 100 ms before the 32-bit counter wraps and ends 200 ms after it. Use what you learned from <code>wraparound.c</code> in Reading 36-1. The answer is one line.

### Step 2 - <code>is_stale</code> (Tests 4 to 8)

Return 1 if a value time-stamped at <code>stamp</code> is more than <code>max_age_ms</code> old at time <code>now</code>, and 0 otherwise. Use your <code>elapsed_ms</code> function. Do not compare <code>now</code> and <code>stamp</code> directly.

<b>Before you go on:</b> Test 8 gives a time stamp 10 ms in the <i>future</i>. What does <code>elapsed_ms</code> return for it, and why is rejecting it the right answer? What could cause a value to arrive time-stamped in the future?

### Step 3 - <code>clock_read</code> (Tests 9 to 11)

Read the 64-bit counter from <code>ticks_hi</code> and <code>ticks_lo</code> using the bounded re-read from Reading 36-2:
<ol>
    <li>Return -1 if <code>now</code> is <code>NULL</code> (Power of Ten Rule 7). You will need <code>#include &lt;stddef.h&gt;</code>.</li>
    <li>Load the high half, call <code>interrupt_point()</code>, then load the low half. <b>The call must go between the two loads</b>: that is where the tests trigger the timer interrupt.</li>
    <li>If the high half is unchanged, combine the halves into <code>*now</code> and return 0.</li>
    <li>Otherwise, try again, up to <code>MAX_READ_TRIES</code> times in total. If the counter never settles, return -1.</li>
</ol>

Test 10 triggers the interrupt one millisecond before the low half wraps, exactly the case from <code>torn_read.c</code>. Test 11 simulates a <b>broken</b> timer whose high half changes on every read. A <code>do-while</code> loop with no bound would hang forever on Test 11.

### Step 4 - <code>to_flight_time</code> (Tests 12 to 15)

Convert a 64-bit millisecond count to whole seconds plus a binary fraction in units of 1/65536 s, as in Method 3 of <code>tick_drift.c</code>:
<ul>
    <li>The seconds are the whole number of seconds in <code>ms</code>.</li>
    <li>The fraction is the leftover milliseconds, converted to units of 1/65536 s. Truncate it; do not round.</li>
    <li>Use only integer arithmetic. No <code>float</code> or <code>double</code>.</li>
</ul>

Watch the order of your casts. Test 15 uses a count just past 2<sup>32</sup> ms, 49.7 days. If you cast <code>ms</code> to 32 bits <i>before</i> dividing, the clock resets to zero on day 50.

### Discussion

<ol>
    <li>Step 1 was one line, and it is the most important line in this program. Why is it the naive comparison, and not the correct subtraction, that most programmers write first?</li>
    <li>In Test 11, <code>clock_read</code> returns -1. What should the flight software do next? Is "keep using the last good time" safe? For how long?</li>
    <li>Test 13 converts 100 ms to 6553 instead of the exact 6553.6. Does that cause drift like Method 2 in <code>tick_drift.c</code>? Why or why not?</li>
</ol>

## Compilation Run

    make test
    make analyze

When you are finished, all 15 tests print <code>PASS</code>, and <code>make analyze</code> reports no errors.